<div style='background:linear-gradient(135deg,#07101d 0%,#0b1626 60%,#0e2a3d 100%);padding:30px 34px;border-radius:16px;border-left:6px solid #22d3ee'>
<div style='color:#22d3ee;font:600 12px/1.4 system-ui,sans-serif;letter-spacing:.16em;text-transform:uppercase'>NFL Big Data Bowl 2027 · Milestone 13 · Colab GPU research notebook</div>
<div style='color:#eaf2fb;font:700 30px/1.25 system-ui,sans-serif;margin:8px 0 12px'>NFL Outcome Transfer, Reliability-Aware Biomechanics &amp; Scouting Value</div>
<div style='color:#b8c7d9;font:400 15px/1.6 system-ui,sans-serif'><b style='color:#eaf2fb'>Primary question.</b> Can 10 Hz NFL Combine movement tracking improve prediction of future NFL movement capacity beyond traditional Combine measurements, under rigorous athlete-disjoint validation?</div>
<div style='color:#7f93ab;font:400 12.5px/1.6 system-ui,sans-serif;margin-top:12px'>Bundle m12f_v1_clean (4867be08f8191b3a) · science 0281883accece2f0 · FP32 · M12 inductive encoders reused read-only · no neural training by default</div>
</div>

## How this notebook works

**Secondary questions.** Which tracking-derived biomechanical metrics give reliable incremental information? Why did F2–F6 fail to retrieve athlete identity across domains, and is predictive information present despite weak retrieval? How much of the predictive performance is position? Which athletes benefit or suffer from tracking-based predictions? Which findings are strong enough for exploratory scouting and player-development analyses?

| Part | Content | Writes to `tables/` |
|---|---|---|
| A | M12 reproducibility and hand-off: environment, checkpoint inventory, reconciliation, prerequisites | `handoff/` |
| B | The prediction problem: frozen-target provenance, eligibility, position structure | `target/` |
| C | Official transfer ladder: POS · M1 · M1+F0 · M1+F1 (if valid) · M1+F2…F6 | `transfer/` |
| D | Common-cohort fairness audit (matched athletes, targets, folds) | `transfer/` |
| E | Reliability-aware biomechanics: ICC audit, equal-observation curves, train-fold-only selection, stability, interactions | `reliability/` |
| F | Position-conditioned capacity: centred, standardised, position-specific, partially pooled | `position/` |
| G | Domain-shift robustness of the frozen embeddings (matching on sequence covariates) | `domain/` |
| H | Why retrieval does not guarantee transfer | `retrieval_transfer/` |
| I | Explanations, help / hurt analysis, intervals, athlete profiles | `scouting/` |
| J | Unified comparison and the declared outcome rule (A / B / C / D / inconclusive) | `synthesis/` |
| K | Executive dashboard, Markdown + HTML report, inventory, export | `synthesis/`, `reports/` |

**Run order.** Setup 1 → restart the session if nflbowl was already imported → Setup 2 … to the end. Every section is a guarded stage (`manifests/stages.json`): a failure is logged with its traceback in `logs/` and later independent sections still run. After a Colab disconnect, rerun from Setup 2 with the same `experiment_name`: adopted checkpoints, embeddings, the ladder record and signature-checked tables are reused.

**Safeguards.** Every performance claim is made on the strict common cohort (identical athletes, targets and folds for every model). Selection thresholds use training athletes only. Encoders missing on any fold are excluded, never replaced by transductive representations. Reports are generated from saved tables only.

In [ ]:
# Setup 1 — install nflbowl from GitHub (Torch is never installed: the Colab CUDA build is kept)
REPO = 'git+https://github.com/mybenkhadda/NFL_bowl.git@main'
%pip install -q {REPO}
%pip install -q --upgrade --force-reinstall --no-deps {REPO}
%pip install -q markdown

In [ ]:
# Setup 2 — version check (after a reinstall: Runtime → Restart session, then continue from here)
import nflbowl
print('nflbowl', nflbowl.__version__)
assert tuple(int(x) for x in nflbowl.__version__.split('.')[:2]) >= (0, 5), 'nflbowl < 0.5.0 is imported: restart the session and rerun from Setup 2'

In [ ]:
# Setup 3 — configuration (unknown keys are errors; neural training needs TWO explicit opt-ins and is off)
CFG = {
    'experiment_name': 'm13_outcome_transfer_v1',
    'bundle_dir': '/content/drive/MyDrive/nfl-big-data-bowl-2027/bundles/m12f_v1_clean',
    'drive_root': '/content/drive/MyDrive/NFLBowl',
    'm12_run_dir': '/content/drive/MyDrive/NFLBowl/experimental_lab/m12_comprehensive_lab_v1',
    'm12_experiment_name': 'm12_comprehensive_lab_v1',
    'stage_dir': '/content/nflbowl_stage',
    'precision': 'fp32',
    'seed': 42,
    'encoders': ['F2', 'F3', 'F4', 'F5', 'F6'],
    'include_f1': True,
    'run_training': False,
    'allow_new_training': False,
    'allow_inference': True,
    'adopt_embeddings': True,
    'verify_checkpoint_load': True,
    'n_boot': 2000,
    'reliability_repeats': 20,
    'reliability_threshold': 0.6,
    'run_domain': True,
    'run_interactions': True,
    'figure_theme': 'navy',
    'extra': {'freeze_json': None},
}
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('not running on Colab: every path in CFG must exist locally')
from nflbowl.lab import m13_config
cfg = m13_config.resolve(CFG)
print('RUN_DIR        ', cfg.run_dir)
print('M12 experiment ', cfg.m12_experiment_dir)
print('training       ', 'ENABLED (explicit opt-in)' if cfg.training_enabled() else 'disabled — M12 encoders are reused')

In [ ]:
# Setup 4 — imports, navy theme, run directory and display helpers
import json, math
import numpy as np
import polars as pl
from IPython.display import display, HTML, Markdown
from nflbowl.lab import (handoff, target_audit, transfer_fairness as TF, reliability as REL, position_models as PM, domain_matching as DM,
                         scouting_value as SV, m13_reporting as REP, m13_figures as M13F, figures as F, biomech, interpret, retrieval_lab, environment)
from nflbowl.lab.store import StageSkipped
from nflbowl.lab.stats import pseudonym
from nflbowl.data.validation import M1_COLS, TRANSFER_TARGET

THEME = M13F.apply_theme(cfg.figure_theme)
RUN = m13_config.M13Run(cfg.run_dir, cfg, local_tmp='/content/m13_tmp')
R = RUN.results
SALT = cfg.experiment_name
NBOOT_FAST = max(200, min(cfg.n_boot, 1000))      # secondary bootstraps (reliability CIs, subgroup reads)
BIO = ['M1+BIO_all', 'M1+BIO_reliable', 'M1+BIO_trainsel', 'M1+BIO_poscond']

def banner(part, title, question):
    display(HTML(M13F.banner_html(part, title, question)))

def say(text, kind='note'):
    display(HTML(M13F.callout_html(text, kind)))

def show(df, digits=4, max_rows=40, signed=(), status=None):
    s = M13F.styled(df, digits, max_rows, signed, status)
    display(s if s is not None else Markdown('_no rows_'))

def check(cond, msg):
    # scientific safeguard: an invalid comparison stops the section (logged as failed by run_stage)
    if not cond:
        raise TF.ComparisonError(msg)

def fig(f, name, data, question, caption):
    return RUN.figure(f, name, data=data, question=question, caption=caption)

def anon(df, col='nfl_id'):
    if df is None or col not in df.columns:
        return df
    return df.with_columns(pl.col(col).map_elements(lambda i: pseudonym(i, SALT), return_dtype=pl.String).alias('athlete'))

def signature(*parts):
    return json.loads(json.dumps(parts, default=str))

print('theme', THEME['theme'], '· run dir', RUN.root)

---
# Part A — M12 reproducibility and hand-off

M13 opens its **own** experiment directory (`RUN_DIR/checkpoints/experiments/m13_outcome_transfer_v1`) under the identical science configuration and **adopts** the M12 inductive checkpoints by copying them (the M12 directory is only read). A checkpoint is valid only if bundle fingerprint, science hash, regime, fold identity and architecture match and the file loads. Embedding files are adopted only when their dependency record names the same weights checksum.

In [ ]:
banner('A1', 'Environment, bundle and provenance', 'Is this session bound to the same bundle, science configuration, precision and seeds as the M12 lab?')

def _a1(ctx):
    global exp, ENV
    ENV = handoff.environment(cfg)
    checks, bsum = environment.validate_bundle(cfg.lab())
    RUN.table(checks, 'bundle_checks', 'handoff')
    show(checks, status='status')
    check(checks.filter(pl.col('status') == 'fail').height == 0, 'bundle checks failed: M13 refuses to run on other data')
    exp = handoff.open_experiment(cfg, RUN)
    rev = ENV.get('revision') or {}
    RUN.set_provenance(bundle_fingerprint=exp.bundle.fingerprint, science_hash=exp.cfg.science_hash(), science_seed=exp.cfg.science.seed,
                       m13_seed=cfg.seed, code_revision=rev.get('commit'), nflbowl=nflbowl.__version__, precision=cfg.precision)
    RUN.write_config(cfg, ENV)
    return bsum

BSUM = RUN.run_stage('A1', _a1, note='environment + bundle + experiment', raise_errors=True)
print('experiment', exp.id, '· science', exp.cfg.science_hash(), '· bundle', exp.bundle.fingerprint)
print('GPU', (ENV.get('gpu') or {}).get('device'), '· precision', cfg.precision, '· revision', (ENV.get('revision') or {}).get('commit'))

In [ ]:
banner('A2', 'M12 checkpoint inventory and read-only adoption', 'Which (model, fold) inductive encoders of M12 are valid for reuse, and which are missing?')

def _a2(ctx):
    folds = sorted(set(int(v) for v in exp.folds_by_athlete.values()))
    src = handoff.discover(cfg, exp)
    if src.height:
        RUN.table(src, 'checkpoint_sources', 'handoff')
    adopted = handoff.adopt(exp, src, list(cfg.encoders)) if src.height else pl.DataFrame()
    emb = handoff.adopt_embeddings(cfg, exp) if cfg.adopt_embeddings else pl.DataFrame()
    inv = handoff.inventory(cfg, exp, ['F0', *cfg.encoders], folds, verify_load=cfg.verify_checkpoint_load)
    ready, missing = handoff.ready_models(inv, folds)
    if missing and cfg.training_enabled():
        say('Training explicitly enabled (two opt-ins): training ONLY the missing inductive encoders.', 'warning')
        tr = environment.train_missing(exp, list(missing), folds, resume=True)
        RUN.table(tr, 'new_training_runs', 'handoff')
        inv = handoff.inventory(cfg, exp, ['F0', *cfg.encoders], folds, verify_load=cfg.verify_checkpoint_load)
        ready, missing = handoff.ready_models(inv, folds)
    f1v, f1t = handoff.f1_validity(exp)
    for name, df in (('adoption', adopted), ('embedding_adoption', emb), ('checkpoint_inventory', inv), ('f1_validity_by_fold', f1t)):
        if df is not None and df.height:
            RUN.table(df, name, 'handoff')
    RUN.save_json(f1v, 'f1_validity')
    return folds, ready, missing, f1v, inv, adopted, emb

FOLDS, READY, MISSING, F1V, INV, ADOPTED, EMB_ADOPTED = RUN.run_stage('A2', _a2, note='discover · adopt · inventory', raise_errors=True)
R.update({'missing_encoders': MISSING, 'f1_validity': F1V})
show(ADOPTED, status='action')
show(EMB_ADOPTED, status='action', max_rows=30)
show(INV.select('model', 'fold', 'training_regime', 'checkpoint_found', 'checkpoint_valid', 'science_hash', 'bundle_fingerprint', 'embedding_found',
                'evaluation_found', 'ready_for_transfer', 'reason'), max_rows=60)
INV_PLOT = INV.with_columns(pl.col('ready_for_transfer').cast(pl.Float64).alias('ready'), pl.col('model').map_elements(F.short, return_dtype=pl.String))
fig(F.heatmap(INV_PLOT, 'model', 'fold', 'ready', 'M12 → M13 readiness matrix', fmt='{:.0f}', vmin=0.0, vmax=1.0, subtitle='1 = valid inductive checkpoint + embedding route for that outer fold'),
    'a2_readiness_matrix', {'inventory': INV}, 'Which (model, fold) encoders can enter the transfer ladder?', 'Readiness of each inductive encoder per outer fold after read-only adoption.')
print('ready on every fold:', READY)
print('missing           :', MISSING or 'none')
print('F1 TCN validity   :', F1V.get('verdict'), '—', F1V.get('detail'))
if MISSING:
    say('Encoders with missing folds are excluded from the ladder and from every dependent experiment (no transductive fallback): '
        + ', '.join(f'{k} (folds {v})' for k, v in MISSING.items()), 'warning')
else:
    say('Every requested encoder has a valid inductive checkpoint on every outer fold.', 'good')

In [ ]:
banner('A3', 'M12 result reconciliation', 'Do the saved M12 artifacts reproduce the values published in the reviewed M12 notebook?')

def _a3(ctx):
    m12, found = handoff.load_m12_tables(cfg)
    rec = handoff.reconcile(m12)
    RUN.table(found, 'm12_tables_found', 'handoff')
    RUN.table(rec, 'm12_reconciliation', 'handoff')
    return m12, found, rec

M12, M12_FOUND, RECON = RUN.run_stage('A3', _a3, note='reload M12 tables (read-only)') or ({}, pl.DataFrame(), pl.DataFrame())
R['reconciliation'] = RECON
show(M12_FOUND, max_rows=30)
show(RECON, status='status')
n_diff = RECON.filter(pl.col('status') != 'consistent').height if RECON.height else 0
say(f'{RECON.height - n_diff} of {RECON.height} reconciled quantities match the published M12 summary within tolerance; {n_diff} differ or are missing. '
    'A difference is reported, never overwritten by the published number: every M13 analysis recomputes from saved predictions and embeddings.',
    'good' if (n_diff == 0 and RECON.height) else 'warning')

In [ ]:
banner('A4', 'Research-gate prerequisites', 'Can the NFL outcome-transfer experiments proceed, and what is missing (and why is it needed)?')
PRE = handoff.prerequisites(exp, INV, READY, MISSING, F1V, M12_FOUND)
RUN.table(PRE, 'prerequisites', 'handoff')
R['prerequisites'] = PRE
show(PRE, status='status', max_rows=30)
HAND = handoff.handoff_record(RUN, cfg, exp, ENV, INV, READY)
n_m12 = M12_FOUND.filter(pl.col('found')).height if M12_FOUND.height else 0
PROV = pl.DataFrame({'item': ['nflbowl version', 'code revision (pip direct_url)', 'bundle fingerprint', 'science hash', 'science seed / M13 seed', 'GPU',
                              'precision', 'M12 experiment (checkpoint source)', 'M13 experiment', 'encoders ready on all folds', 'outer folds', 'M12 tables found'],
                     'value': [nflbowl.__version__, str((ENV.get('revision') or {}).get('commit')), exp.bundle.fingerprint, exp.cfg.science_hash(),
                               f'{exp.cfg.science.seed} / {cfg.seed}', str((ENV.get('gpu') or {}).get('device')), cfg.precision, str(cfg.m12_experiment_dir),
                               str(exp.root), ', '.join(READY) or 'none', ', '.join(map(str, FOLDS)), str(n_m12)]})
RUN.table(PROV, 'provenance', 'handoff')
R['provenance_table'] = PROV
R['handoff_inventory_summary'] = INV.group_by('model').agg(pl.col('ready_for_transfer').sum().alias('folds_ready'), pl.len().alias('folds')).sort('model')
show(PROV)
BLOCKING = ['bundle fingerprint = declared', 'science hash = declared', 'athletes.parquet present', f'frozen target {TRANSFER_TARGET}', 'M1 official columns', 'fold manifest']
blocking = PRE.filter((pl.col('status') == 'MISSING') & pl.col('requirement').is_in(BLOCKING))
check(blocking.height == 0, 'blocking prerequisites missing: ' + ', '.join(blocking['requirement'].to_list()))
say('All blocking prerequisites pass. Non-blocking gaps (missing encoders, F1 validity) restrict which rungs run; they are listed above and repeated in the automated caveats.', 'good')

---
# Part B — The NFL prediction problem

The outcome is the frozen 12C target `nfl.q90_shared`: the speed-conditioned τ = 0.90 acceleration quantile averaged over shared speed bins, from all NFL games of an athlete. It is a **constructed movement-capacity quantity**, not an NFL success outcome. B1 separates athlete-disjoint *prediction* validation from the *target estimator's* fitting scope; B2 counts who is eligible; B3 measures how much of the target is position.

In [ ]:
banner('B1', 'Frozen target audit', 'What exactly is nfl.q90_shared, how was it estimated, and which claims does its provenance allow?')
FREEZE = target_audit.read_freeze(cfg.extra.get('freeze_json'))
TPROV = RUN.run_stage('B1', lambda ctx: target_audit.target_provenance(exp, FREEZE), note='target provenance')
if TPROV is not None:
    RUN.table(TPROV, 'target_provenance', 'target')
    R['target_provenance'] = TPROV
    show(TPROV, max_rows=20)
say('Prediction validation is athlete-disjoint, but the target estimator, its shared speed bins and (if C3_pooled was frozen) its position pooling were '
    'fitted once on all athletes before any Combine value was joined. M13 therefore claims athlete-disjoint PREDICTION validity only, never strictly '
    'inductive end-to-end target construction.', 'warning')

In [ ]:
banner('B2', 'Sample eligibility', 'How many athletes survive each requirement, and how do the eligible sets overlap?')

def _b2(ctx):
    sets = target_audit.athlete_sets(exp)
    flow = target_audit.attrition_flow(sets)
    ov = target_audit.overlap_matrix(sets)
    RUN.table(flow, 'attrition_flow_pre_ladder', 'target')
    RUN.table(ov, 'overlap_matrix_pre_ladder', 'target')
    fig(F.waterfall(flow, 'Cohort selection flow (before the ladder)', 'athletes remaining after each sequential requirement'), 'b2_cohort_flow', {'flow': flow},
        'How many athletes survive each requirement?', 'Sequential attrition: bundle → NFL tracking → linked → frozen target → fold → complete M1 → Combine forty tracking.')
    fig(F.heatmap(ov, 'set_a', 'set_b', 'share_of_a', 'Sample-overlap matrix', fmt='{:.2f}', vmin=0.0, vmax=1.0, subtitle='share of the row set contained in the column set'),
        'b2_overlap_matrix', {'overlap': ov}, 'How do the eligible athlete sets overlap?', 'Share of each row set contained in each column set.')
    return flow

FLOW0 = RUN.run_stage('B2', _b2, note='attrition before the ladder')
show(FLOW0)

In [ ]:
banner('B3', 'Position structure of the target', 'How much of the target variance lies between positions, i.e. how predictable is it from position alone?')

def _b3(ctx):
    af = target_audit.analysis_frame(exp)
    vd = target_audit.variance_decomposition(af['y'].to_numpy(), af['position_group'].to_numpy())
    pt = target_audit.position_table(af, exp, cfg.min_position_n)
    out = target_audit.outliers(af)
    RUN.table(pl.DataFrame([vd]), 'variance_decomposition', 'target')
    RUN.table(pt, 'position_table', 'target')
    if out.height:
        RUN.table(anon(out), 'target_outliers', 'target')
    plot = af.with_columns(pl.lit('frozen target').alias('cohort'))
    fig(F.distribution_compare(plot, 'y', 'cohort', 'NFL target by position', 'nfl.q90_shared (yd/s²)', facet='position_group', max_facets=12,
                               subtitle='violin + IQR + median per position · right: pooled ECDF'), 'b3_target_by_position',
        {'target': plot.select('position_group', 'y')}, 'How is the frozen target distributed within and between positions?', 'Distribution of nfl.q90_shared by position group (yd/s²).')
    fig(F.hbar(pt.sort('n', descending=True), 'position_group', 'n', 'Athletes with the frozen target per position', 'athletes'), 'b3_position_counts', {'positions': pt},
        'How many athletes support each position?', 'Athletes in the ladder population by position group.')
    return af, vd, pt

AF, VD, PT = RUN.run_stage('B3', _b3, note='position structure') or (None, {}, None)
R.update({'variance': pl.DataFrame([VD]) if VD else None, 'position_table': PT})
if VD:
    show(pl.DataFrame([VD]))
    show(PT, max_rows=20)
    eta = VD.get('eta2', float('nan'))
    say(f'Between-position variance is {eta:.1%} of the total target variance (in-sample η²). Every ladder rung contains unpenalized position dummies and '
        'inherits this predictability; Part C therefore reports the out-of-fold position-only model (POS) and skill beyond position for every model.')

---
# Part C — The official NFL transfer ladder

`Experiment.transfer_predictions` (unchanged ladder): athlete outer folds, train-fold-only design, fold-fitted embedding PCA (≤ 8 dims), inner-CV ridge α, position as unpenalized dummies. M13 adds **POS** (position dummies only) and **MEAN** (training-fold mean) through the *same* engine on the *same* frame; the baseline run re-fits M1 and must reproduce the ladder M1 exactly (assertion). All metrics are computed through the Part D fairness system: one validated long prediction table and explicit cohorts.

In [ ]:
banner('C1', 'Official NFL transfer ladder', 'Do inductive tracking representations lower out-of-fold RMSE of the frozen target beyond the official Combine measurements?')

def _c1(ctx):
    oof = TF.ladder_predictions(exp, READY, reuse=cfg.reuse_artifacts)
    base, data = TF.baseline_predictions(exp)
    repro = TF.reproducibility_check(oof, base)
    check(repro['status'] == 'pass', f'baseline-run M1 does not reproduce the ladder M1: {repro}')
    RUN.data_file(oof, 'datasets/cached/ladder_oof', {'models': READY, 'kind': 'transfer_oof_v1'})
    RUN.data_file(base, 'datasets/cached/baseline_oof', {'kind': 'baselines_v1'})
    RUN.table(pl.DataFrame([repro]), 'reproducibility_check', 'transfer')
    return oof, base, data, repro

OOF, BASE, DATA, REPRO = RUN.run_stage('C1', _c1, note='ladder + POS / MEAN baselines', raise_errors=True)
F0IDS = TF.f0_available(DATA)
F1_OK = bool(cfg.include_f1 and F1V.get('verdict') == 'valid' and 'pred_M1+F1_TCN' in OOF.columns)
ENC_MODELS = [f'M1+{n}' for n in READY]
PRIMARY = ['POS', 'M1', 'M1+F0', *ENC_MODELS] + (['M1+F1_TCN'] if F1_OK else [])
SECONDARY = ['M1+F1_TCN'] if ('pred_M1+F1_TCN' in OOF.columns and not F1_OK) else []
R['repro_check'] = REPRO
print('ladder models :', [c.removeprefix('pred_') for c in OOF.columns if c.startswith('pred_')])
print('primary models:', PRIMARY)
print('secondary     :', SECONDARY or 'none')
show(pl.DataFrame([REPRO]), status='status')

In [ ]:
banner('C2', 'Common-cohort assembly (Part D system)', 'Are all compared predictions aligned on the same athletes, targets and folds?')

def _c2(ctx):
    ladder_models = [c.removeprefix('pred_') for c in OOF.columns if c.startswith('pred_')]
    keep = [m for m in ladder_models if m in PRIMARY or m in SECONDARY]
    long = pl.concat([TF.long_predictions(OOF, 'ladder', models=keep), TF.long_predictions(BASE, 'baselines', models=['POS', 'MEAN'])], how='vertical')
    audit = TF.validate_alignment(long, exp.folds_by_athlete)
    coh, desc = TF.cohorts(long, OOF, PRIMARY, F0IDS, READY)
    n_strict = len(coh['strict_common'])
    check(n_strict >= 20, f'strict common cohort has only {n_strict} athletes: no valid paired interval')
    return long, audit, coh, desc

LONG, ALIGN, COH, COH_DESC = RUN.run_stage('C2', _c2, note='alignment + cohorts', raise_errors=True)
R['cohorts_desc'] = COH_DESC
show(ALIGN, status='status')
show(COH_DESC)

In [ ]:
banner('C3', 'Ladder metrics and paired comparisons', 'On identical athletes, which representation changes RMSE relative to M1 and to M1+F0, and how stable is that across folds?')
MODELS_C = [*PRIMARY, 'MEAN']

def _c3(ctx):
    out = {}
    pairs, prim = TF.comparison_pairs(MODELS_C, READY)
    for name in ('strict_common', 'ladder_all'):
        w = TF.wide(LONG, COH[name], MODELS_C)
        out[name] = {'w': w, 'metrics': TF.cohort_metrics(w, MODELS_C, name),
                     'paired': TF.paired_comparisons(w, pairs, cfg.n_boot, cfg.seed, name, primary_family=prim),
                     'rmse_ci': TF.rmse_intervals(w, MODELS_C, cfg.n_boot, cfg.seed, name), 'folds': TF.fold_table(w, MODELS_C)}
    native = []
    for m in [*PRIMARY, *SECONDARY]:
        ids = TF.native_cohort(m, OOF, F0IDS, COH['ladder_all'])
        if len(ids) >= 20:
            w1 = TF.wide(LONG, ids, list(dict.fromkeys([m, 'POS'])))
            native.append(TF.cohort_metrics(w1, [m], 'native:' + m).with_columns(pl.lit(False).alias('comparable_across_models')))
    nat = pl.concat(native, how='diagonal_relaxed') if native else pl.DataFrame()
    s, a = out['strict_common'], out['ladder_all']
    for n, d in (('metrics_strict_common', s['metrics']), ('metrics_ladder_all', a['metrics']), ('metrics_native', nat),
                 ('rmse_intervals', pl.concat([s['rmse_ci'], a['rmse_ci']], how='vertical')), ('fold_metrics_strict_common', s['folds']),
                 ('fold_metrics_ladder_all', a['folds'])):
        if d is not None and d.height:
            RUN.table(d, n, 'transfer')
    paired = pl.concat([s['paired'], a['paired']], how='diagonal_relaxed')
    RUN.table(paired, 'paired_transfer_comparisons', 'transfer')
    RUN.table(s['w'], 'common_cohort_predictions', 'transfer')
    max_diff = REPRO.get('max_abs_pred_diff')
    audit = pl.concat([ALIGN.with_columns(pl.lit('alignment').alias('section')),
                       COH_DESC.select(pl.col('cohort').alias('check'), pl.lit('info').alias('status'), pl.col('definition').alias('detail'),
                                       pl.col('n_athletes').alias('n'), pl.lit('cohort').alias('section')),
                       pl.DataFrame([{'check': REPRO['check'], 'status': REPRO['status'], 'detail': f'max abs prediction difference {max_diff}', 'n': None,
                                      'section': 'reproducibility'}])], how='diagonal_relaxed')
    RUN.table(audit, 'common_cohort_audit', 'transfer')
    return out, nat, paired, audit

C, NATIVE, PAIRED_C, CC_AUDIT = RUN.run_stage('C3', _c3, note='metrics + paired bootstrap', raise_errors=True)
W_S, M_S, FT_S = C['strict_common']['w'], C['strict_common']['metrics'], C['strict_common']['folds']
PS = PAIRED_C.filter((pl.col('cohort') == 'strict_common') & (pl.col('reference') == 'M1'))
R.update({'metrics_strict': M_S, 'paired_strict': PAIRED_C.filter(pl.col('cohort') == 'strict_common'), 'metrics_native': NATIVE, 'common_cohort_audit': CC_AUDIT})
check(set(PAIRED_C.filter(pl.col('cohort') == 'strict_common')['n_athletes'].to_list()) == {W_S.height}, 'a strict-cohort comparison used a different number of athletes')
show(M_S.select('model', 'n_athletes', 'rmse', 'mae', 'r2', 'r2_within_position', 'skill_vs_position_oof', 'mean_error', 'spearman', 'fold_rmse_sd'),
     signed=('skill_vs_position_oof',))
show(PAIRED_C.filter(pl.col('cohort') == 'strict_common').select('model', 'reference', 'n_athletes', 'delta_rmse', 'delta_lo', 'delta_hi', 'delta_pct',
                                                                  'p_boot_le0', 'p_holm_primary', 'verdict'), signed=('delta_rmse',), status='verdict', max_rows=60)
say(REP.read_paired(PAIRED_C, 'strict_common', 'M1'))
say(REP.read_paired(PAIRED_C, 'strict_common', 'M1+F0'))

In [ ]:
banner('C4', 'Ladder figures', 'Where do the models differ: overall, per fold, per position and per athlete?')
enc_rank = PS.filter(pl.col('model').is_in(ENC_MODELS)).sort('delta_rmse', descending=True)['model'].to_list()
BEST_ENC = enc_rank[0] if enc_rank else None
FOCUS = [m for m in ['M1', 'M1+F0', BEST_ENC] if m]

def _c4(ctx):
    tgt = pl.concat([target_audit.analysis_frame(exp).select('y').with_columns(pl.lit('ladder population').alias('cohort')),
                     W_S.select('y').with_columns(pl.lit('strict common cohort').alias('cohort'))], how='vertical')
    fig(F.histogram_by(tgt, 'y', 'cohort', 'NFL target distribution', 'nfl.q90_shared (yd/s²)'), 'c4_target_distribution', {'target': tgt},
        'Is the strict cohort representative of the ladder population?', 'Frozen target in the ladder population and in the strict common cohort.')
    fig(F.pred_vs_actual(W_S, PRIMARY, 'Prediction vs actual (strict common cohort)', 'out-of-fold predictions · dashed = identity'), 'c4_pred_vs_actual',
        {'predictions': W_S.drop('nfl_id')}, 'How close are OOF predictions to the observed target?', 'Out-of-fold prediction vs observed nfl.q90_shared per model.')
    rci = C['strict_common']['rmse_ci']
    fig(F.forest(rci, 'model', 'rmse', 'rmse_lo', 'rmse_hi', 'Model RMSE (strict common cohort)', 'RMSE (yd/s²)', color_by='model', subtitle='95 % fold-stratified athlete bootstrap'),
        'c4_rmse_comparison', {'rmse': rci}, 'Which model has the lowest held-out RMSE?', 'RMSE per model with bootstrap intervals; identical athletes in every row.')
    for ref, tag in (('M1', 'm1'), ('M1+F0', 'f0')):
        d = PAIRED_C.filter((pl.col('cohort') == 'strict_common') & (pl.col('reference') == ref))
        fig(F.forest(d, 'model', 'delta_rmse', 'delta_lo', 'delta_hi', f'Δ RMSE vs {ref}', 'Δ RMSE = RMSE(reference) − RMSE(model) (yd/s²)', ref=0.0, ref_label='no difference',
                     color_by='model', subtitle='positive = better than the reference · 95 % paired bootstrap · strict common cohort'),
            f'c4_delta_rmse_forest_vs_{tag}', {'paired': d}, f'Does each model beat {ref} on identical athletes?', f'Paired Δ RMSE vs {ref} with 95 % intervals.')
    fts = FT_S.with_columns(pl.col('model').map_elements(F.short, return_dtype=pl.String))
    fig(F.heatmap(fts, 'model', 'fold', 'rmse', 'Fold-by-fold RMSE', fmt='{:.3f}', subtitle='strict common cohort · yd/s²'), 'c4_fold_rmse_heatmap', {'folds': FT_S},
        'Is the model ranking stable across outer folds?', 'RMSE per model and outer fold.')
    res = TF.residual_long(W_S, PRIMARY)
    rf = res.filter(pl.col('model').is_in(FOCUS))
    fig(F.residual_panels(res, FOCUS, 'Residual diagnostics', 'left: residual ECDF · right: median residual by observed-target decile'), 'c4_residuals',
        {'residuals': rf.drop('nfl_id')}, 'Where do models err, e.g. in the tails of the target?', 'Residual density and residual versus target.')
    fig(F.distribution_compare(rf, 'residual', 'model', 'Residual by position', 'residual (yd/s²)', facet='position_group', max_facets=12, order=FOCUS),
        'c4_residual_by_position', {'residuals': rf.drop('nfl_id')}, 'Are errors position-specific?', 'Residual distribution per position for M1, M1+F0 and the best encoder.')
    ra = TF.rank_agreement(W_S, PRIMARY)
    RUN.table(ra, 'rank_agreement', 'transfer')
    rap = ra.with_columns(pl.col('model_a').map_elements(F.short, return_dtype=pl.String), pl.col('model_b').map_elements(F.short, return_dtype=pl.String))
    fig(F.heatmap(rap, 'model_a', 'model_b', 'spearman', 'Predicted ranking agreement', fmt='{:.2f}', vmin=0.0, vmax=1.0, subtitle='Spearman between OOF predictions'),
        'c4_rank_agreement', {'agreement': ra}, 'Do models rank athletes the same way?', 'Spearman correlation between model predictions.')
    if BEST_ENC:
        bw = anon(TF.best_worst(W_S, BEST_ENC, 'M1', 8))
        RUN.table(bw, 'best_worst_cases', 'transfer')
        fig(M13F.case_dumbbell(bw, 'Best / worst prediction changes: ' + F.short(BEST_ENC) + ' vs M1', 'absolute error of M1 (open) → of the encoder model (filled)'),
            'c4_best_worst_cases', {'cases': bw.drop('nfl_id')}, 'For which athletes does the tracking model help or hurt most?', 'Largest absolute-error improvements and deteriorations (pseudonymised).')
    fs = FT_S.filter(pl.col('model').is_in(ENC_MODELS + ['M1+F0'])).group_by('model').agg((pl.col('delta_rmse_vs_M1') > 0).sum().alias('folds_better_than_M1'),
                                                                                      pl.len().alias('folds')).sort('model')
    return fs

FS = RUN.run_stage('C4', _c4, note='ladder figures')
show(FS)
say('Fold stability: a representation that beats M1 in only some outer folds is not a stable improvement, whatever its pooled point estimate.')

---
# Part D — Common-cohort fairness audit

The M12 notebook reported a 270-athlete complete-M1 cohort next to a 291-athlete interpretable-model prediction cohort. Such metrics are not directly comparable. The M13 system joins predictions by athlete and fold, validates target and fold equality and uniqueness, builds explicit cohorts and recomputes every metric on them. Native-coverage metrics are kept separate.

In [ ]:
banner('D1', 'Fairness audit', 'Are all M13 performance claims made on matched samples, and what explains differing cohort sizes?')
show(CC_AUDIT, status='status', max_rows=40)
la, st = len(COH['ladder_all']), len(COH['strict_common'])
n_m1inc = int(COH_DESC.filter(pl.col('cohort') == 'excluded from strict (M1 incomplete)')['n_athletes'][0])
say(f'The ladder scores {la} athletes (every athlete with the frozen target; M1 / F0 / embedding gaps are imputed inside the design with training medians). '
    f'{n_m1inc} of them lack at least one M1 measurement, which is exactly why a complete-M1 count and a prediction count differ (the M12 270 vs 291 discrepancy is of this kind). '
    f'Every M13 claim uses the strict common cohort of {st} athletes; native-coverage numbers are shown separately and never compared across models.', 'warning')
CMP = M_S.select('model', pl.col('n_athletes').alias('n_strict'), pl.col('rmse').alias('rmse_strict'))
if NATIVE.height:
    CMP = CMP.join(NATIVE.select('model', pl.col('n_athletes').alias('n_native'), pl.col('rmse').alias('rmse_native')), on='model', how='left')
RUN.table(CMP, 'native_vs_common', 'transfer')
show(CMP)

In [ ]:
# D2 — scientific safeguards: every claim below this cell relies on these assertions
check(W_S.height == len(COH['strict_common']), 'strict cohort frame size differs from the cohort definition')
check(W_S['nfl_id'].n_unique() == W_S.height, 'duplicate athletes in the strict cohort')
fmap = exp.folds_by_athlete
check(all(int(fmap[int(i)]) == int(f) for i, f in zip(W_S['nfl_id'].to_list(), W_S['fold'].to_list())), 'fold drift in the strict cohort')
check(set(PS['n_athletes'].to_list()) == {W_S.height}, 'a paired comparison used an unmatched sample')
check(all(r['status'] == 'pass' for r in ALIGN.iter_rows(named=True)), 'alignment audit not clean')
print('fairness assertions: all passed')

In [ ]:
banner('D3', 'Complete attrition flow', 'How is the strict common cohort reached, including native embedding coverage?')

def _d3(ctx):
    sets = target_audit.athlete_sets(exp, OOF, READY)
    flow = target_audit.attrition_flow(sets, READY)
    last = int(flow['n_remaining'][-1]) if flow.height else 0
    flow = pl.concat([flow, pl.DataFrame([{'step': 'strict common cohort', 'n_remaining': len(COH['strict_common']),
                                          'n_removed': max(0, last - len(COH['strict_common']))}])], how='vertical')
    sets['strict common cohort'] = COH['strict_common']
    ov = target_audit.overlap_matrix(sets)
    RUN.table(flow, 'attrition_flow', 'target')
    RUN.table(ov, 'overlap_matrix', 'target')
    fig(F.waterfall(flow, 'Cohort selection flow', 'sequential requirements down to the strict common cohort'), 'd_cohort_flow', {'flow': flow},
        'How is the strict common cohort reached?', 'Sequential attrition including the native embedding coverage of every ready encoder.')
    fig(F.heatmap(ov, 'set_a', 'set_b', 'n_overlap', 'Sample-overlap matrix (athletes)', fmt='{:.0f}'), 'd_overlap_matrix', {'overlap': ov},
        'Which eligibility sets overlap?', 'Number of athletes in both sets.')
    return flow

FLOW = RUN.run_stage('D3', _d3, note='full attrition')
R['attrition'] = FLOW
show(FLOW)

---
# Part E — Reliability-aware biomechanical features

**Question.** Are stable, physically interpretable tracking signals more useful than generic learned representations?

Candidates (Combine side only; NFL-window metrics never enter a predictor frame): peak speed, acceleration rate 0→90 %, time to 90 %, peak acceleration, braking impulse, peak deceleration, heading-rate variability, turning at speed, curvature, jerk, movement consistency, turning asymmetry (exploratory: sign convention unverified). E1 audits the M12 split-half estimator and replaces it by an unclamped one-way ICC with variance components; E2 compares domains at equal trial counts; E3 selects features with **training athletes only**; E4 checks stability; E5 tests declared interactions (exploratory).

In [ ]:
banner('E0', 'Biomechanical feature table', 'Which interpretable Combine movement metrics are available per athlete for the ladder population?')

def _e0(ctx):
    seq, how = handoff.sequence_metrics(exp, RUN, cfg)
    ath = biomech.athlete_features(seq)
    RUN.data_file(ath, 'features/athlete/athlete_movement_features', {'metric_version': biomech.VERSION})
    bd = biomech.ladder_frame(exp, ath)
    cand = REL.candidate_columns(bd)
    check(not any(c.startswith('nfl__') for c in cand), 'NFL-window metrics leaked into the candidate predictors')
    check(set(OOF['nfl_id'].to_list()) <= set(bd['nfl_id'].to_list()), 'biomechanics frame lacks ladder athletes')
    RUN.table(biomech.metric_definitions(), 'metric_definitions', 'reliability')
    print('sequence metrics:', how, '·', seq.height, 'sequences ·', len(cand), 'candidate Combine columns')
    return seq, ath, bd, cand

SEQ, ATHF, BDATA, CAND = RUN.run_stage('E0', _e0, note='features', raise_errors=True)
show(pl.DataFrame({'candidate': CAND, 'coverage': [float(BDATA[c].is_not_null().mean()) for c in CAND]}), max_rows=40)

In [ ]:
banner('E1', 'Reliability audit', 'Are the M12 split-half reliabilities trustworthy, and where do the extreme negative values come from?')

def _e1(ctx):
    sg = signature('E1', CAND, NBOOT_FAST, cfg.seed, cfg.min_reliability_athletes)
    hit = RUN.cached('reliability', ['reliability_audit'], sg)
    if hit:
        ctx['cache'] = 'hit'
        aud = hit['reliability_audit']
    else:
        aud = REL.reliability_audit(SEQ, CAND, NBOOT_FAST, cfg.seed, cfg.min_reliability_athletes)
        RUN.save_all({'reliability_audit': aud}, 'reliability', sg)
    expl = REL.explain_m12(M12.get('biomechanics/repeatability'), aud)
    if expl.height:
        RUN.table(expl, 'm12_negative_reliability_explained', 'reliability')
    return aud, expl

AUD, EXPL = RUN.run_stage('E1', _e1, note='ICC audit') or (pl.DataFrame(), pl.DataFrame())
if AUD.height:
    R['reliability_audit_summary'] = AUD.select('feature', 'domain', 'scope', 'n_athletes', 'r_half', 'sb_raw', 'icc1', 'icc1_lo', 'icc1_hi')
    show(AUD.select('feature', 'domain', 'scope', 'n_athletes', 'trials_per_athlete_median', 'r_half', 'sb_raw', 'icc1', 'icc1_lo', 'icc1_hi', 'icc1_lower_bound',
                    'var_between_raw', 'var_within', 'n_identical_pairs', 'flags'), max_rows=60, signed=('icc1',))
    show(EXPL, max_rows=30)
    sc = AUD.filter(pl.col('sb_raw').is_finite() & pl.col('icc1').is_finite())
    fig(F.scatter(sc, 'sb_raw', 'icc1', 'Split-half Spearman–Brown vs ICC(1,1)', 'Spearman–Brown 2r/(1+r) (M12 estimator, unclamped)', 'ICC(1,1) (one-way, all trials)',
                  group='domain', identity=True, ref_x=0.0, ref_y=0.0, subtitle='SB is unbounded below for r < 0; ICC is bounded by −1/(k0−1)'),
        'e1_sb_vs_icc', {'audit': sc}, 'Do the two estimators agree, and where does SB explode?', 'Each point is one metric × domain × scope.')
    comb = AUD.filter((pl.col('domain') == 'combine') & pl.col('scope').is_in(['forty', 'pooled drills (kind-centred)']))
    fig(F.forest(comb, 'feature', 'icc1', 'icc1_lo', 'icc1_hi', 'Combine repeatability ICC(1,1)', 'ICC(1,1)', ref=cfg.reliability_threshold, ref_label='selection threshold',
                 subtitle='athlete-cluster bootstrap 95 % · forty attempts / kind-centred agility drills'),
        'e1_icc_forest', {'icc': comb}, 'Which metrics are repeatable across Combine attempts?', 'ICC(1,1) per candidate column with the declared selection threshold.')
    n_neg = int((AUD['r_half'].fill_nan(None).drop_nulls() < 0).sum())
    say(f'{n_neg} metric × domain rows have a negative split-half correlation. For those rows the M12 estimator 2r/(1+r) is not a reliability (it is unbounded below), '
        'which explains the extreme negative M12 values. M13 reports the one-way ICC with its variance components and theoretical bound instead, unclamped.',
        'warning' if n_neg else 'note')

In [ ]:
banner('E2', 'Equal-observation reliability', 'With the same athletes and the same number of trials per domain, are Combine and NFL metrics equally repeatable?')

def _e2(ctx):
    sg = signature('E2', CAND, cfg.reliability_k_grid, cfg.reliability_repeats, cfg.seed, cfg.min_reliability_athletes)
    hit = RUN.cached('reliability', ['reliability_curves', 'reliability_curve_differences'], sg)
    if hit:
        ctx['cache'] = 'hit'
        return hit['reliability_curves'], hit['reliability_curve_differences']
    cur, dif = REL.equal_observation_curves(SEQ, CAND, cfg.reliability_k_grid, cfg.reliability_repeats, cfg.seed, cfg.min_reliability_athletes)
    RUN.save_all({'reliability_curves': cur, 'reliability_curve_differences': dif}, 'reliability', sg)
    return cur, dif

CURVES, DIFFS = RUN.run_stage('E2', _e2, note='matched-k curves') or (pl.DataFrame(), pl.DataFrame())
PROJ = REL.projected_curve(AUD.filter(pl.col('scope').is_in(['forty', 'pooled drills (kind-centred)']))) if AUD.height else None
fig(M13F.reliability_curves(CURVES, PROJ, 'Reliability vs matched trial count', 'same athletes and same k per domain · band = 2.5–97.5 % over seeded repeats'),
    'e2_reliability_curves', {'curves': CURVES, 'projected': PROJ}, 'Does repeatability differ by domain at equal observation counts?', 'ICC(1,1) versus matched trials per athlete.')
if DIFFS.height:
    DSUM = DIFFS.group_by('feature', 'k').agg(pl.col('diff_nfl_minus_combine').mean().alias('mean'), pl.col('diff_nfl_minus_combine').quantile(0.025).alias('q025'),
                                              pl.col('diff_nfl_minus_combine').quantile(0.975).alias('q975'), pl.len().alias('repeats')).sort('feature', 'k')
    RUN.table(DSUM, 'reliability_domain_difference_summary', 'reliability')
    show(DSUM, signed=('mean',), max_rows=40)
    say('Differences are NFL − Combine ICC(1,1) at equal k on the same athletes; the 2.5–97.5 % range is over sampling repeats (trial draws), not a population interval.')
else:
    say('No feature had enough athletes with the same number of trials in both domains (Combine attempts per drill are few): equal-observation curves are not estimable.', 'warning')

In [ ]:
banner('E3', 'Reliability-aware feature selection', 'Do stable, interpretable Combine metrics add out-of-fold value beyond M1 when every selection threshold is learned on training athletes only?')

def _e3(ctx):
    cache = REL.trial_cache(SEQ, CAND)
    boof, sel, blog = REL.selection_ladder(exp, BDATA, CAND, cache, cfg.reliability_threshold, cfg.train_selected_k, cfg.min_reliability_athletes)
    j = OOF.select('nfl_id', pl.col('pred_M1').alias('a')).join(boof.select('nfl_id', pl.col('pred_M1').alias('b')), on='nfl_id')
    diff = float((j['a'] - j['b']).abs().max())
    check(j.height == OOF.height and diff < 1e-8, f'biomechanics ladder M1 differs from the official M1 (max {diff:.2e})')
    RUN.data_file(boof, 'datasets/cached/biomech_selection_oof', {'threshold': cfg.reliability_threshold, 'k': cfg.train_selected_k})
    for n, d in (('selection_log', sel), ('selection_ladder_log', blog)):
        if d.height:
            RUN.table(d, n, 'reliability')
    return boof, sel, cache

BOOF, SEL, TCACHE = RUN.run_stage('E3', _e3, note='train-fold selection ladders') or (None, pl.DataFrame(), {})
if BOOF is not None:
    LONG = pl.concat([LONG.filter(pl.col('source') != 'biomech'), TF.long_predictions(BOOF, 'biomech', models=BIO)], how='vertical')
    TF.validate_alignment(LONG, exp.folds_by_athlete)
    W_B = TF.wide(LONG, COH['strict_common'], ['POS', 'M1', 'M1+F0', *BIO])
    pairs = [(m, 'M1') for m in BIO] + [(m, 'M1+BIO_all') for m in BIO if m != 'M1+BIO_all'] + [(m, 'M1+F0') for m in BIO]
    P_BIO = TF.paired_comparisons(W_B, pairs, cfg.n_boot, cfg.seed, 'strict_common')
    RUN.table(P_BIO, 'selection_paired_comparisons', 'reliability')
    R['selection_paired'] = P_BIO
    show(P_BIO.select('model', 'reference', 'n_athletes', 'delta_rmse', 'delta_lo', 'delta_hi', 'p_boot_le0', 'verdict'), signed=('delta_rmse',), status='verdict')
    fig(F.forest(P_BIO.filter(pl.col('reference') == 'M1'), 'model', 'delta_rmse', 'delta_lo', 'delta_hi', 'Biomechanics vs M1 (strict common cohort)', 'Δ RMSE (yd/s²)',
                 ref=0.0, ref_label='no difference', color_by='model', subtitle='train-fold-only selection · positive = better than M1'),
        'e3_biomech_forest', {'paired': P_BIO}, 'Do reliability-filtered biomechanics beat M1?', 'Paired Δ RMSE vs M1 of the four biomechanics feature sets.')
    fig(M13F.selection_heatmap(SEL, 'Train-fold feature selection', '1 = selected in that outer fold by the reliability or train-association rule'),
        'e3_selection_heatmap', {'selection': SEL}, 'Which metrics are selected, and how stable is the selection across folds?', 'Selection indicator per feature, model and outer fold.')
    say(REP.read_paired(P_BIO, 'strict_common', 'M1'))

In [ ]:
banner('E4', 'Feature stability', 'Are the candidate metrics stable across trials, folds and positions, distinct from the officials, and do they carry residual signal after M1?')

def _e4(ctx):
    stb = REL.feature_stability(BDATA, TCACHE, CAND, cfg.min_reliability_athletes, cfg.min_position_n)
    inc = biomech.incremental_value(exp, BDATA, CAND, cfg.seed, n_boot=NBOOT_FAST)
    RUN.table(stb, 'feature_stability', 'reliability')
    if inc.height:
        RUN.table(inc, 'incremental_value', 'reliability')
    return stb, inc

STAB, INC = RUN.run_stage('E4', _e4, note='stability + incremental value') or (pl.DataFrame(), pl.DataFrame())
show(STAB, max_rows=40, signed=('partial_assoc_train_mean',))
if INC.height:
    inc_p = INC.with_columns(pl.col('metric').str.replace('c_forty__', 'forty·', literal=True).str.replace('c_agility__', 'agility·', literal=True).alias('label'))
    fig(F.forest(inc_p, 'label', 'delta_rmse', 'delta_lo', 'delta_hi', 'Added-feature importance (one metric at a time)', 'Δ RMSE vs M1 (yd/s²)', ref=0.0, ref_label='no difference',
                 subtitle='OOF ladder M1 vs M1 + metric · all ladder rows (design-internal imputation) · BH-adjusted in the table'),
        'e4_added_feature_importance', {'incremental': INC}, 'Which single metric adds most beyond M1?', 'Δ RMSE of adding each metric to M1 (one at a time).')
say('In-sample partial associations in the stability table are descriptive only; selection and every Δ RMSE use out-of-fold fits.')

In [ ]:
banner('E5', 'Feature interactions (exploratory)', 'Do declared movement interactions add beyond their additive terms?')

def _e5(ctx):
    it, io = REL.interaction_tests(exp, BDATA, NBOOT_FAST, cfg.seed)
    if it.height:
        RUN.table(it, 'interaction_tests', 'reliability')
    return it

INT = RUN.run_stage('E5', _e5, enabled=cfg.run_interactions, reason_disabled='run_interactions=False')
if INT is not None and INT.height:
    R['interactions'] = INT
    show(INT.select('interaction', 'cohort', 'n_athletes', 'delta_rmse', 'delta_lo', 'delta_hi', 'p_boot_le0', 'p_bh_family', 'verdict'), signed=('delta_rmse',), status='verdict')
    fig(F.forest(INT, 'interaction', 'delta_rmse', 'delta_lo', 'delta_hi', 'Interaction vs additive model', 'Δ RMSE (yd/s²)', ref=0.0, ref_label='no difference',
                 subtitle='exploratory · train-fold standardisation inside the block · BH across the family'),
        'e5_interactions', {'interactions': INT}, 'Do movement interactions add predictive value?', 'Paired Δ RMSE of each interaction model vs its additive counterpart.')
say('Interaction results are exploratory associations of a predictive model, not causal effects; they do not enter the decision rule.', 'warning')

---
# Part F — Position-conditioned NFL capacity

Pooled R² may be dominated by position. For three feature sets (M1, M1+F0, M1+BIO_all) M13 fits five variants per outer fold on training athletes only: pooled (= ladder), target centred on the training position mean, target standardised with training position mean / SD, position-specific ridge, and prediction-level partial pooling with κ chosen by inner CV. Original-scale RMSE and within-position metrics are both reported; small position groups get no interval.

In [ ]:
banner('F', 'Position-conditioned models', 'How much predictive skill remains within positions, and do position-specific or partially pooled models help?')

def _f(ctx):
    Cm = exp.cfg.science.to_m12config()
    eng = [c for c in DATA.columns if c.startswith('eng')]
    sets = {'M1': (DATA, list(M1_COLS)), 'M1+F0': (DATA, [*M1_COLS, *eng]), 'M1+BIO_all': (BDATA, [*M1_COLS, *CAND])}
    oofs, logs = [], []
    for tag, (frame, cols) in sets.items():
        o, lg = PM.oof_position_models(frame, cols, tag, Cm, Cm.seed, kappas=cfg.partial_pooling_kappas)
        oofs.append(o)
        logs.append(lg)
    return oofs, pl.concat(logs, how='diagonal_relaxed')

POUT = RUN.run_stage('F1', _f, enabled=cfg.run_position_models, reason_disabled='run_position_models=False')
if POUT:
    POOFS, PLOG = POUT
    RUN.table(PLOG, 'position_model_log', 'position')
    LONG = pl.concat([LONG.filter(pl.col('source') != 'position'), *[TF.long_predictions(o, 'position') for o in POOFS]], how='vertical')
    TF.validate_alignment(LONG, exp.folds_by_athlete)
    PMODELS = [c.removeprefix('pred_') for o in POOFS for c in o.columns if c.startswith('pred_')]
    W_P = TF.wide(LONG, COH['strict_common'], ['POS', 'M1', *PMODELS])
    MP = TF.cohort_metrics(W_P, ['POS', 'M1', *PMODELS], 'strict_common').join(PM.within_position_metrics(W_P, ['POS', 'M1', *PMODELS]), on='model', how='left')
    pairs = [(m, 'M1') for m in PMODELS] + [(m, m.split('·')[0] + '·pooled') for m in PMODELS if not m.endswith('·pooled')]
    P_POS = TF.paired_comparisons(W_P, pairs, cfg.n_boot, cfg.seed, 'strict_common')
    RUN.table(MP, 'position_model_metrics', 'position')
    RUN.table(P_POS, 'position_model_paired', 'position')
    R['position_paired'] = P_POS
    show(MP.select('model', 'rmse', 'r2', 'r2_within_position', 'skill_vs_position_oof', 'rmse_position_sd_units', 'mean_within_position_r2'), signed=('skill_vs_position_oof',))
    show(P_POS.select('model', 'reference', 'delta_rmse', 'delta_lo', 'delta_hi', 'verdict'), signed=('delta_rmse',), status='verdict', max_rows=40)
    best_p = P_POS.filter(pl.col('reference') == 'M1').sort('delta_rmse', descending=True)['model'][0]
    PE = PM.position_effects(W_P, best_p, 'M1', NBOOT_FAST, cfg.seed, cfg.min_position_n, cfg.adequate_n)
    RUN.table(PE, 'position_effects', 'position')
    pe = PE.filter(pl.col('reliability') != 'insufficient').with_columns(
        pl.concat_str([pl.col('position_group'), pl.lit(' (n='), pl.col('n_athletes').cast(pl.String), pl.lit(', '), pl.col('reliability'), pl.lit(')')]).alias('label'))
    fig(F.forest(pe, 'label', 'delta_rmse', 'delta_lo', 'delta_hi', 'Within-position effect of ' + best_p + ' vs M1', 'Δ RMSE (yd/s²)', ref=0.0, ref_label='no difference',
                 subtitle='groups below the minimum size are suppressed · exploratory below the adequate size'),
        'f_position_effects', {'effects': PE}, 'Which positions gain from the best position-conditioned model?', 'Per-position paired Δ RMSE with sample sizes and reliability labels.')
    fig(F.forest(MP, 'model', 'skill_vs_position_oof', None, None, 'Skill beyond position (out-of-fold)', '1 − SSE(model) / SSE(position-only OOF)', ref=0.0, ref_label='position only'),
        'f_skill_beyond_position', {'metrics': MP}, 'How much skill do models have beyond the position-only baseline?', 'Out-of-fold skill relative to the position-only model.')
    say(REP.read_paired(P_POS, 'strict_common', 'M1'))

---
# Part G — Domain-shift robustness

Sensitivity analyses on the **frozen** inductive embeddings of each outer fold (no encoder is trained or re-embedded). Held-out sequences are matched across domains by coarsened exact matching on label-free sequence covariates (valid frames, duration, peak speed, peak acceleration / |a_n|); position subsets are evaluated separately. Models that would need re-embedding or retraining are listed, not run.

In [ ]:
banner('G', 'Domain-shift robustness', 'Is the near-perfect Combine / NFL separability explained by sequence length, duration, position, speed or movement state?')
G_FOLDS = list(cfg.domain_folds) if cfg.domain_folds else FOLDS
G_MODELS = ['F0_engineered', *READY]

def _g(ctx):
    sg = signature('G', G_MODELS, G_FOLDS, cfg.domain_bins, cfg.seed, cfg.domain_max_per_domain)
    hit = RUN.cached('domain', ['domain_matching_long', 'domain_matching_summary'], sg)
    if hit:
        ctx['cache'] = 'hit'
        return hit['domain_matching_long'], hit['domain_matching_summary']
    dl, ds = DM.run(exp, SEQ, G_MODELS, G_FOLDS, cfg.domain_bins, cfg.seed, NBOOT_FAST, cfg.domain_max_per_domain)
    RUN.save_all({'domain_matching_long': dl, 'domain_matching_summary': ds}, 'domain', sg)
    return dl, ds

GOUT = RUN.run_stage('G1', _g, enabled=cfg.run_domain, reason_disabled='run_domain=False')
DL, DS = GOUT if GOUT else (pl.DataFrame(), pl.DataFrame())
if DS.height:
    R['domain_summary'] = DS
    show(DS.select('representation', 'condition', 'domain_auc_grouped', 'domain_auc_min', 'domain_auc_max', 'delta_auc_vs_original', 'mmd2', 'delta_mmd2_vs_original',
                   'norm_ratio_nfl_over_combine', 'retrieval_auc', 'n_rows', 'n_folds'), signed=('delta_auc_vs_original',), max_rows=60)
    dsp = DS.with_columns(pl.col('representation').map_elements(F.short, return_dtype=pl.String))
    fig(F.heatmap(dsp, 'representation', 'condition', 'domain_auc_grouped', 'Domain AUC after matching', fmt='{:.3f}', vmin=0.5, vmax=1.0,
                  subtitle='grouped domain-classifier AUC · mean over folds · frozen embeddings'),
        'g_domain_auc_heatmap', {'summary': DS}, 'Does matching on sequence covariates remove domain separability?', 'Domain AUC per representation and matching condition.')
    fig(F.heatmap(dsp, 'representation', 'condition', 'mmd2', 'MMD² after matching', fmt='{:.3f}', subtitle='RBF MMD² (median heuristic) · mean over folds'),
        'g_mmd2_heatmap', {'summary': DS}, 'How large is the distribution gap under each matching?', 'MMD² per representation and condition.')
    fig(F.heatmap(dsp, 'representation', 'condition', 'norm_ratio_nfl_over_combine', 'Embedding norm ratio NFL / Combine', fmt='{:.2f}', diverging=True, center=1.0),
        'g_norm_ratio', {'summary': DS}, 'Do the domains differ in embedding scale?', 'Mean embedding norm ratio per condition (1 = equal).')
    fig(F.heatmap(dsp, 'representation', 'condition', 'retrieval_auc', 'Cross-domain retrieval AUC among matched sequences', fmt='{:.3f}', diverging=True, center=0.5),
        'g_retrieval_sensitivity', {'summary': DS}, 'Does matching change identity retrieval?', 'Retrieval AUC on matched held-out sequences (0.5 = chance).')
    CD = DM.conditional_distributions(SEQ)
    fig(M13F.covariate_ecdfs(CD, 'Sequence covariates by domain', 'the distributions the matching acts on (all sequences, subsampled for plotting)', seed=cfg.seed),
        'g_conditional_distributions', {'covariates': CD.sample(min(CD.height, 200000), seed=cfg.seed)}, 'How different are Combine and NFL sequences before matching?',
        'ECDF of each sequence covariate per domain.')
    show(pl.DataFrame({'not run here (needs re-embedding or retraining)': list(DM.RETRAINING_REQUIRED)}))
    orig = float(DS.filter(pl.col('condition') == 'original')['domain_auc_grouped'].mean())
    drop = float(DS.filter(pl.col('condition') != 'original')['delta_auc_vs_original'].min())
    say(f'Mean original domain AUC {orig:.3f}; the largest change under any matching is {drop:+.3f}. Matching uses only sequence covariates of held-out rows (no outcome labels). '
        'A domain AUC that stays near 1 after matching means the separation is not explained by that covariate.')

---
# Part H — Why retrieval does not guarantee transfer

Per (representation, fold): inductive retrieval AUC, effective rank, domain AUC, MMD², M12 probe summaries (when saved) and the fold-level Δ RMSE of the ladder on the strict cohort. Associations are read across models (few points, exact permutation p) and within models across folds after two-way demeaning. Neither metric is assumed to cause the other.

In [ ]:
banner('H', 'Retrieval versus transfer', 'Does better identity retrieval (or geometry / domain gap) of a representation go with better NFL target prediction?')

def _h(ctx):
    models = ['F0_engineered', *READY]
    ret = SV.retrieval_records(exp, models, FOLDS)
    geo = SV.geometry_records(exp, models, FOLDS)
    probes = SV.probe_summary(M12.get('ablations/inductive_probes'))
    tr = SV.transfer_records(FT_S).with_columns(pl.col('representation_short').replace({'F0': 'F0_engineered'}))
    rel = SV.relationship_table(ret, geo, DL if DL.height else None, tr, probes)
    asc = SV.associations(rel)
    for n, d in (('retrieval_records', ret), ('geometry_records', geo), ('relationship_table', rel), ('associations', asc)):
        if d.height:
            RUN.table(d, n, 'retrieval_transfer')
    return ret, rel, asc

HOUT = RUN.run_stage('H1', _h, note='retrieval vs transfer')
RET, RELT, ASSOC = HOUT if HOUT else (pl.DataFrame(), pl.DataFrame(), pl.DataFrame())
if RELT.height:
    R['associations'] = ASSOC
    show(RELT, max_rows=40, signed=('retrieval_gain_vs_F0', 'delta_rmse_vs_M1', 'delta_rmse_vs_M1F0'))
    show(ASSOC)
    fig(F.scatter(RELT.filter(pl.col('representation') != 'F0_engineered'), 'retrieval_gain_vs_F0', 'delta_rmse_vs_M1F0', 'Retrieval gain vs transfer gain (model × fold)',
                  'retrieval AUC − F0 AUC (same fold)', 'Δ RMSE vs M1+F0 (yd/s²)', group='representation', ref_x=0.0, ref_y=0.0,
                  subtitle='association only: neither metric is assumed to cause the other'),
        'h_retrieval_vs_transfer', {'records': RELT}, 'Does improved retrieval correspond to improved target prediction?', 'Each point is one representation in one outer fold.')
    ml = RELT.group_by('representation').agg(pl.col('retrieval_auc').mean(), pl.col('delta_rmse_vs_M1').mean(), pl.col('effective_rank').mean())
    fig(F.scatter(ml, 'retrieval_auc', 'delta_rmse_vs_M1', 'Model-level retrieval AUC vs Δ RMSE vs M1', 'inductive retrieval AUC (mean over folds)', 'Δ RMSE vs M1 (mean over folds)',
                  group='representation', ref_x=0.5, ref_y=0.0), 'h_model_level', {'models': ml}, 'Across representations, does retrieval rank models the way transfer does?',
        'One point per representation (means over folds).')
    say('Associations are computed across models (few points, exact permutation p-values) and within models across folds after two-way demeaning; folds of one model are '
        'never treated as independent architectures. A weak or absent association means retrieval quality is not a usable proxy for transfer here.')

---
# Part I — Model explanations and scouting insights

The model explained is the one with the largest Δ RMSE vs M1 among those whose interval excludes 0 on the strict cohort; if none qualifies, the largest point estimate is shown **for exploration only** and labelled as not validated. Out-of-fold predictions only. Interval coverage is measured before any interval is shown. The target is a constructed movement-capacity quantity: nothing here predicts NFL success.

In [ ]:
banner('I', 'Explanations, help / hurt analysis and athlete profiles', 'For the best validated model, where does tracking help or hurt, and what does it reveal about athletes with similar stopwatch times?')
P_SO_FAR = pl.concat([x for x in (PAIRED_C.filter(pl.col('cohort') == 'strict_common'), R.get('selection_paired'), R.get('position_paired')) if x is not None],
                     how='diagonal_relaxed')
BEST = SV.best_model(P_SO_FAR, 'strict_common', 'M1')
BEST_MODEL, BEST_OK = BEST.get('model'), BEST.get('validated')
say(f'Model explained: {BEST_MODEL} · validated: {BEST_OK} · rule: ' + str(BEST.get('reason')), 'good' if BEST_OK else 'warning')

def _i(ctx):
    if BEST_MODEL is None:
        raise StageSkipped('blocked: no comparison rows to choose a model from')
    models_i = list(dict.fromkeys(['POS', 'M1', 'M1+F0', *ENC_MODELS, BEST_MODEL]))
    w = TF.wide(LONG, COH['strict_common'], models_i)
    gain = SV.tracking_gain(w, BEST_MODEL, 'M1')
    gbg = SV.gain_by_group(gain, SV.athlete_context(exp, ATHF), NBOOT_FAST, cfg.seed, cfg.min_position_n)
    twins, cells = SV.forty_twins(w, ['POS', 'M1', BEST_MODEL], BDATA)
    unexp = SV.unexplained_by_position(w, ['POS', 'M1', BEST_MODEL], cfg.min_position_n)
    intervals, ivc = SV.oof_intervals(w, BEST_MODEL, cfg.interval_level)
    relc = REL.column_reliability(AUD) if AUD.height else {}
    reliable = [c for c in CAND if np.isfinite(relc.get(c, float('nan'))) and relc[c] >= cfg.reliability_threshold] or CAND[:6]
    il = interpret.interpretable_ladder(BDATA, reliable, exp, cfg.seed, n_repeats=10)
    sel = SV.select_profiles(gain, cfg.n_profiles, cfg.seed)
    idx = SV.build_profiles(RUN, w, sel, BEST_MODEL, 'M1', BDATA, reliable, intervals, ivc, SALT, models_i, relc)
    for n, d in (('tracking_gain', anon(gain)), ('gain_by_group', gbg), ('forty_twins', twins), ('forty_twin_cells', cells), ('unexplained_by_position', unexp),
                 ('oof_intervals', anon(intervals)), ('interpretable_importance', il['importance']), ('interpretable_coefficients', il['coefficients']),
                 ('interpretable_pdp', il['pdp']), ('athlete_profile_index', idx)):
        if d is not None and d.height:
            RUN.table(d, n, 'scouting')
    RUN.save_json(ivc, 'interval_check')
    return {'w': w, 'gain': gain, 'gbg': gbg, 'twins': twins, 'cells': cells, 'unexp': unexp, 'ivc': ivc, 'il': il, 'idx': idx, 'reliable': reliable, 'sel': sel}

I = RUN.run_stage('I1', _i, enabled=cfg.run_scouting, reason_disabled='run_scouting=False')
if I:
    R.update({'best_model': BEST, 'gain_by_group': I['gbg'], 'interval_check': I['ivc']})
    show(I['gbg'], signed=('mean_abs_error_gain',), max_rows=40)
    show(I['twins'])
    show(I['unexp'].filter(~pl.col('small_group')), max_rows=40)
    print('interval check:', {k: v for k, v in I['ivc'].items() if k != 'by_position'})
    show(I['idx'].drop('path'))

In [ ]:
# I — figures
if I:
    g = I['gbg'].filter(pl.col('variable') == 'position_group').with_columns(
        pl.concat_str([pl.col('level'), pl.lit(' (n='), pl.col('n').cast(pl.String), pl.lit(')')]).alias('label'))
    fig(F.forest(g, 'label', 'mean_abs_error_gain', 'lo', 'hi', 'Where tracking helps or hurts (by position)', 'mean |error| gain vs M1 (yd/s²)', ref=0.0, ref_label='no change',
                 subtitle=F.short(BEST_MODEL) + ' vs M1 · positive = tracking model closer to the target · athlete bootstrap'),
        'i_help_hurt_by_position', {'gain_by_group': I['gbg']}, 'In which positions does tracking help or hurt?', 'Mean absolute-error gain over M1 per position.')
    fig(F.hbar(I['twins'], 'model', 'within_cell_r2', 'Athletes with similar forty times', 'pooled within-cell R² (position × forty quartile)', color_by='model', fmt='{:.3f}',
               subtitle='how much target variance each model explains among stopwatch twins'),
        'i_forty_twins', {'twins': I['twins'], 'cells': I['cells']}, 'Does tracking separate athletes the stopwatch cannot?', 'Within-cell R² among athletes of the same position and forty quartile.')
    imp = I['il']['importance'].filter(pl.col('position_group') == 'ALL')
    if imp.height:
        impp = imp.with_columns(pl.col('feature').str.replace('c_forty__', 'forty·', literal=True).str.replace('c_agility__', 'agility·', literal=True))
        fig(F.hbar(impp, 'feature', 'importance_delta_rmse', 'Interpretable model: permutation importance', 'Δ RMSE when the feature is permuted (yd/s²)', fmt='{:.4f}',
                   subtitle='ridge(M1 + reliable metrics) · out-of-fold · predictive, not causal'),
            'i_interpretable_importance', {'importance': imp}, 'Which inputs carry the interpretable model?', 'Permutation importance of officials and reliable movement metrics.')
    bw = anon(TF.best_worst(I['w'], BEST_MODEL, 'M1', 6))
    fig(M13F.case_dumbbell(bw, 'Case studies: ' + F.short(BEST_MODEL) + ' vs M1', 'absolute error of M1 (open) → of the explained model (filled)'),
        'i_case_errors', {'cases': bw.drop('nfl_id')}, 'Which athletes gain or lose most?', 'Largest absolute-error changes (pseudonymised).')
    ids = [int(a) for a in I['sel']['nfl_id'].to_list()[:6]]
    labels = {a: pseudonym(a, SALT) + ' · ' + s for a, s in zip(ids, I['sel']['selection'].to_list()[:6])}
    prof = retrieval_lab.speed_profiles(exp, ids, labels)
    if prof.height:
        prof = prof.with_columns(pl.col('label').alias('athlete'))
        fig(F.profile_panels(prof, 'Athlete case studies: movement profiles', 'mean speed of Combine forty attempts and NFL windows (± 1 SD) · yd/s'),
            'i_case_profiles', {'profiles': prof}, 'What do the selected athletes move like?', 'Speed profiles of the profiled athletes (pseudonymised).')
    say('Profiles are research views of a constructed movement-capacity quantity (reports/athlete_profiles/). Out-of-fold predictions only; intervals are shown only '
        'when their empirical coverage is within 10 points of nominal.', 'warning')

---
# Part J — Decision-relevant scientific synthesis

One validated prediction table (ladder, baselines, biomechanics, position models) → one strict common cohort → one paired bootstrap. The outcome rule is declared in `nflbowl.lab.m13_reporting` before results are read: **A** embeddings improve · **B** biomechanics improve, embeddings do not · **C** both · **D** neither, with precise nulls · otherwise **inconclusive**. Training loss is never used for ranking; small non-significant gains never promote a model.

In [ ]:
banner('J', 'Unified comparison and decision', 'Taken together, which outcome (A–D) does the measured evidence support?')

def _j(ctx):
    models_j = [m for m in LONG['model'].unique(maintain_order=True).to_list() if F1_OK or m != 'M1+F1_TCN']
    w = TF.wide(LONG, COH['strict_common'], models_j)
    met = TF.cohort_metrics(w, models_j, 'strict_common')
    pairs, prim = TF.comparison_pairs(models_j, READY)
    pj = TF.paired_comparisons(w, pairs, cfg.n_boot, cfg.seed, 'strict_common', primary_family=prim)
    ft = TF.fold_table(w, models_j)
    relinfo = {'M1': 'official measurements', 'POS': '—', 'MEAN': '—', 'M1+F0': 'deterministic features'}
    relinfo.update({m: 'learned (reliability n/a)' for m in ENC_MODELS})
    for m in BIO:
        s = SEL.filter(pl.col('model') == m) if SEL.height else SEL
        if s.height:
            k = float(s['n_selected'].mean())
            relinfo[m] = f'{k:.1f} features / fold (train-fold rule)'
    uni = REP.unified_comparison(pj, met, RET if RET.height else None, DS if DS.height else None, relinfo)
    out = REP.determine_outcome(pj, 'strict_common', ENC_MODELS, [m for m in BIO if m in models_j])
    ev = REP.evidence_rows(uni, ft, DS if DS.height else None)
    for n, d in (('final_metrics', met), ('final_paired_comparisons', pj), ('final_fold_metrics', ft), ('unified_comparison', uni), ('evidence_matrix', ev)):
        RUN.table(d, n, 'synthesis')
    RUN.save_json(out, 'outcome')
    RUN.table(LONG, 'all_predictions_long', 'transfer')
    return uni, out, ev, pj, met

UNI, OUTCOME, EVID, P_J, M_J = RUN.run_stage('J1', _j, note='synthesis', raise_errors=True)
R.update({'unified': UNI, 'outcome': OUTCOME})
show(UNI.select('model', 'n_athletes', 'rmse', 'delta_rmse', 'delta_lo', 'delta_hi', 'r2_within_position', 'retrieval_auc', 'domain_auc', 'reliability_evidence', 'verdict'),
     signed=('delta_rmse',), status='verdict', max_rows=40)
fig(M13F.evidence_matrix(EVID, 'model', 'criterion', 'status', 'label', 'Research evidence matrix',
                         'measured values · green = supports · amber = inconclusive · red = against · grey = not evaluated'),
    'j_evidence_matrix', {'evidence': EVID}, 'Which criteria does each model meet?', 'Evidence per model and criterion on the strict common cohort.')
say('Outcome ' + OUTCOME['outcome'] + ' — ' + OUTCOME['title'] + '. ' + OUTCOME['recommendation'] + chr(10) + chr(10).join(OUTCOME['evidence']),
    {'good': 'good', 'warning': 'warning', 'neutral': 'note'}[OUTCOME['status']])

---
# Part K — Executive dashboard, reports and artifacts

In [ ]:
banner('K', 'Executive dashboard and reports', 'What should a reader take away, and where is every artifact?')

def _k(ctx):
    ps = P_J.filter((pl.col('reference') == 'M1') & pl.col('model').is_in([*ENC_MODELS, 'M1+F0', *BIO, 'POS']))
    n_strict, n_all = len(COH['strict_common']), len(COH['ladder_all'])
    m1r = float(M_J.filter(pl.col('model') == 'M1')['rmse'][0])
    tiles = [('Strict common cohort', str(n_strict), f'athletes of {n_all} in the ladder'), ('RMSE of M1', f'{m1r:.3f}', 'yd/s² · official Combine reference'),
             ('Encoders evaluated', str(len(READY)), 'inductive · every outer fold'), ('Outcome', OUTCOME['outcome'], OUTCOME['title'][:44])]
    dash = M13F.dashboard(tiles, ps.sort('delta_rmse'), 'M13 executive dashboard', 'NFL movement-capacity transfer under athlete-disjoint validation',
                          outcome='Outcome ' + OUTCOME['outcome'] + ': ' + OUTCOME['recommendation'], outcome_status=OUTCOME['status'])
    fig(dash, 'k_executive_dashboard', {'forest': ps}, 'What is the decision?', 'KPI tiles, the primary Δ RMSE forest and the declared outcome.')
    figs = pl.DataFrame(R.get('_figures', []), infer_schema_length=None)
    if figs.height:
        RUN.table(figs.select([c for c in figs.columns if not c.startswith('data:')]), 'figure_index', 'synthesis')
    R['figures'] = R.get('_figures', [])
    R['stages'] = REP.stage_summary(RUN)
    md = REP.final_report(RUN, cfg, ENV, R)
    paths = REP.write_reports(RUN, md)
    RUN.table(RUN.inventory(), 'artifact_inventory', 'synthesis')
    z = RUN.export_zip(include=('config', 'manifests', 'tables', 'figures', 'reports', 'logs'))
    return paths, z

KOUT = RUN.run_stage('K1', _k, note='dashboard + reports + export')
if KOUT:
    print('report  :', KOUT[0])
    print('export  :', KOUT[1])
show(REP.stage_summary(RUN).select('stage', 'execution', 'note'), status='execution', max_rows=40)

In [ ]:
# Executive summary (from the measured outcome record) and automated caveats
display(HTML(M13F.banner_html('Summary', 'Outcome ' + OUTCOME['outcome'] + ' — ' + OUTCOME['title'], OUTCOME['recommendation'])))
for line in OUTCOME['evidence']:
    say(line)
for c in REP.caveats(R):
    say(c, 'warning')
print('artifacts under', RUN.root)